# BrickGPT + new 1x3 brick — Colab pipeline (Stage 0 baseline, Stage 3 LoRA, Stage 4 evaluation)

EECS 182 course project. Model: **BrickGPT** = Llama-3.2-1B-Instruct + official LoRA `AvaLovelace/BrickGPT`.
The base model is gated, so we load the ungated mirror `unsloth/Llama-3.2-1B-Instruct` (same weights) and merge the
official adapter on top, exactly like our local `pilot/common.py`.

**Runtime:** GPU (T4 or A100). `Runtime → Change runtime type → GPU`.

| Stage | What | Output (on Drive) |
|---|---|---|
| 0 | Baseline generation on a fixed random subset of test captions, rollback ON vs OFF, original brick library | `stage0/gen.jsonl`, `stage0/scored.jsonl`, `stage0/metrics.json` |
| 3 | LoRA from the released BrickGPT checkpoint, one run per `stage2_data/{group}_N{n}.jsonl`, same hyper-parameters | `stage3/<run>/adapter + train_log.json` |
| 4 | Stage-0 protocol again with each trained adapter and the 1x3 brick in the library | `stage4/<run>/...`, `stage4/summary.json` |

Everything is parameterised in the next cell. Every generated structure is appended to a JSONL immediately, and all
loops **resume** (already-finished items/runs are skipped), so a Colab disconnect only loses the current item.

**Helper code:** the notebook writes `helpers.py` itself (cell "Write helpers.py"); it is an identical copy of
`newbrick/colab/helpers.py` in our repo, so nothing has to be uploaded except the Gurobi licence and the Stage-2 data.

**Runtime estimates.** GPU timings: *unknown* (nothing has been run on a GPU). What we measured locally on CPU:
full generation 2–6 min per object (pilot `gen.py`: 60–126 bricks, `max_regenerations=2`, connectivity check);
one Gurobi stability check 1–53 s per structure (median 3.5 s, 10 test structures, 32–239 bricks); tiny LoRA
250 examples × 1 epoch ≈ 77 min on CPU. The Gurobi check runs on the CPU in Colab too, so it does not get faster
with a GPU. Use the **timing cell** in Stage 0 (a few items) to extrapolate before launching the full run.

In [ ]:
# ============================== PARAMETERS ==============================
# --- storage
USE_DRIVE = True                      # save everything to Google Drive (recommended: Colab sessions die)
DRIVE_DIR = '/content/drive/MyDrive/brickgpt_newbrick'   # output root on Drive
LOCAL_DIR = '/content/brickgpt_newbrick'                 # used if USE_DRIVE=False
MODELS_DIR = '/content/models'        # merged full models (large, ~5 GB each in fp32) -> local disk, not Drive

# --- code / models
BRICKGPT_REPO = 'https://github.com/AvaLovelace1/BrickGPT.git'
BRICKGPT_COMMIT = '9c53c7cb65f9de0df89562cc74b2d2e270fbcab4'  # commit of our local clone (pilot/BrickGPT)
BASE_MODEL = 'unsloth/Llama-3.2-1B-Instruct'   # ungated mirror of meta-llama/Llama-3.2-1B-Instruct
BRICKGPT_ADAPTER = 'AvaLovelace/BrickGPT'
DATASET = 'AvaLovelace/StableText2Brick'

# --- Gurobi
USE_GUROBI = True                     # False -> rollback uses the connectivity check; stability scored later locally
GUROBI_LIC_DRIVE_PATH = '/content/drive/MyDrive/gurobi/gurobi.lic'  # if present it is used, else upload prompt

# --- Stage 0 / Stage 4 generation
N_CAPTIONS = 100                      # size of the fixed random test subset
SUBSET_SEED = 0                       # seed for choosing the subset (recorded in subset.json)
GEN_SEED_BASE = 1000                  # item k is generated with transformers.set_seed(GEN_SEED_BASE + k)
CAPTION_INDEX = 0                     # which of the captions of an object to use (pilot/gen.py used 0)
MAX_BRICKS = 2000                     # BrickGPTConfig default
MAX_REGENERATIONS_ON = 100            # BrickGPTConfig default (rollback ON)
MAX_BRICK_REJECTIONS = 500            # BrickGPTConfig default (per-brick rejection sampling; kept ON in both conds)
ALSO_DISABLE_REJECTION_IN_OFF = False # True -> rollback-OFF also sets max_brick_rejections=0 (no validity filtering)
CONDITIONS = ['rollback_on', 'rollback_off']
TIMING_N = 3                          # items for the timing pilot cell

# --- Stage 3 LoRA
STAGE2_DIR = DRIVE_DIR + '/stage2_data'   # upload newbrick/stage2_data/ here
TRAIN_GROUPS = ['B_best', 'C1', 'C2']     # files {group}_N{n}.jsonl from build_stage2_data.py
TRAIN_SIZES = [100, 500, 2000]
VAL_FILE = 'heldout_B_best.jsonl'         # eval loss / best checkpoint; set None to disable
HP_OVERRIDES = {}                         # e.g. {'num_train_epochs': 1}; same HP for every run (logged)
MAX_STEPS = -1                            # >0 only for a quick smoke test

# --- Stage 4
EVAL_RUNS = None                      # None -> every trained run found in stage3/; or a list of run names
EVAL_CONTROL_RELEASED_WITH_1X3 = True # also evaluate the released model with 1x3 in library + the new prompt
NEW_BRICK_MASS = 0.00119              # midpoint of 1x2 and 1x4 masses; same value as newbrick/stage1.py (an estimate)

## Setup: Drive, dependencies, BrickGPT

We install BrickGPT with `--no-deps`: its `pyproject.toml` requires `bpy` (Blender, only used for rendering) and
pins `numpy<2`, both of which fight with Colab's preinstalled stack. The modules we use (models, data, stability
analysis) only need torch, transformers, peft, gurobipy, networkx, numpy. Versions in BrickGPT's `uv.lock`:
transformers 4.51.3, peft 0.15.2, trl 0.17.0, gurobipy 12.0.2; our working local venv: transformers 4.57.6,
peft 0.21.2, gurobipy 13.0.3. We pin transformers < 5 (untested with 5.x).

In [ ]:
import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
OUT = DRIVE_DIR if USE_DRIVE else LOCAL_DIR
os.makedirs(OUT, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print('outputs ->', OUT)

In [ ]:
!pip install -q "transformers>=4.51,<5" "peft>=0.15" accelerate datasets gurobipy networkx
!test -d /content/BrickGPT || git clone -q {BRICKGPT_REPO} /content/BrickGPT
!cd /content/BrickGPT && git checkout -q {BRICKGPT_COMMIT} && git log -1 --oneline
!pip install -q --no-deps -e /content/BrickGPT
!pip uninstall -y -q torchao  # Colab's old torchao breaks new peft ("incompatible version of torchao"); not needed

## Write helpers.py

Identical copy of `newbrick/colab/helpers.py` (1x3 library patch from `pilot/common.py`, model merging, resumable
generation, scoring, LoRA training). Alternative: upload that file to `/content/helpers.py` and skip this cell.

In [ ]:
%%writefile /content/helpers.py
"""Helpers for the BrickGPT + 1x3-brick Colab notebook (brickgpt_newbrick.ipynb).

Self-contained copy of what the notebook needs from our local code (pilot/common.py, gen.py, lora_tiny.py),
so Colab does not need our repo. Requires the BrickGPT repo on sys.path (``import brickgpt`` must work).

Sections
  1. Brick library patch (1x3 brick, id 99, LDraw part 3622.DAT)          -- from pilot/common.py
  2. Prompts / instruction functions                                        -- from pilot/common.py + brickgpt.py
  3. Model building (released BrickGPT = Llama-3.2-1B-Instruct + LoRA; optional extra adapter merged on top)
  4. Test subset, generation loop (resumable JSONL)
  5. Post-hoc scoring (validity, Gurobi stability or connectivity fallback) + summary metrics
  6. LoRA training (HF Trainer, hyper-parameters of scripts/finetune.zsh by default)
"""
from __future__ import annotations

import json
import os
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np

# ======================================================================================
# 1. Brick library patch
# ======================================================================================
NEW_ID = '99'
NEW_BRICK = {'height': 1, 'width': 3, 'mass': 0.00119, 'inventory': 100000, 'partID': '3622.DAT'}  # midpoint of 1x2 and 1x4, same as newbrick/stage1.py


def _bl_module():
    # NOTE: `import brickgpt.data.brick_library as bl` returns the *dict* (brickgpt.data re-exports a dict named
    # brick_library that shadows the submodule), hence importlib, as in pilot/common.py.
    import importlib
    return importlib.import_module('brickgpt.data.brick_library')


def set_1x3_in_library(enabled: bool, mass: float | None = None) -> None:
    """Add (enabled=True) or remove (False) the 1x3 brick from BrickGPT's in-memory brick library.

    Same patch as pilot/common.py: brick_library['99'] + _dimensions_to_brick_id_dict[(1, 3)] = 99.
    The dict objects are mutated in place, so modules that imported them by name (brick_structure,
    stability analysis) see the change. Nothing is written to disk.
    """
    bl = _bl_module()
    if enabled:
        props = dict(NEW_BRICK)
        if mass is not None:
            props['mass'] = mass
        bl.brick_library[NEW_ID] = props
        bl._dimensions_to_brick_id_dict[(1, 3)] = int(NEW_ID)
    else:
        bl.brick_library.pop(NEW_ID, None)
        if bl._dimensions_to_brick_id_dict.get((1, 3)) == int(NEW_ID):
            del bl._dimensions_to_brick_id_dict[(1, 3)]


def library_has_1x3() -> bool:
    return (1, 3) in _bl_module()._dimensions_to_brick_id_dict


# ======================================================================================
# 2. Prompts
# ======================================================================================
ORIG_DIMS = '2x4, 4x2, 2x6, 6x2, 1x2, 2x1, 1x4, 4x1, 1x6, 6x1, 1x8, 8x1, 1x1, 2x2'
NEW_DIMS = ORIG_DIMS + ', 1x3, 3x1'
SYSTEM_PROMPT = 'You are a helpful assistant.'
INPUT_MARKER = '### Input:\n'


def create_instruction(caption: str, dims: str = ORIG_DIMS) -> str:
    """brickgpt.models.brickgpt.create_instruction with the dimension list parameterised.
    With dims=ORIG_DIMS the output is byte-identical to the original."""
    return ('Create a LEGO model of the input. Format your response as a list of bricks: '
            '<brick dimensions> <brick position>, where the brick position is (x,y,z).\n'
            f'Allowed brick dimensions are {dims}.\n'
            'All bricks are 1 unit tall.\n\n'
            '### Input:\n'
            f'{caption}')


def instruction_fn_from_prefix(prefix: str):
    """Instruction function = fixed prefix (everything up to and including '### Input:\\n') + caption."""
    def fn(caption: str) -> str:
        return prefix + caption
    return fn


def prefix_from_dataset(jsonl_path: str | Path) -> str:
    """Read the first example of a {"messages": [...]} JSONL and return the user-prompt prefix, so evaluation
    uses exactly the prompt the adapter was trained with."""
    with open(jsonl_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                msgs = json.loads(line)['messages']
                user = next(m['content'] for m in msgs if m['role'] == 'user')
                i = user.rfind(INPUT_MARKER)
                if i < 0:
                    raise ValueError(f'No "{INPUT_MARKER!r}" in user prompt of {jsonl_path}')
                return user[:i + len(INPUT_MARKER)]
    raise ValueError(f'{jsonl_path} is empty')


# ======================================================================================
# 3. Model building
# ======================================================================================
def build_merged_model(out_dir: str | Path, base_id: str = 'unsloth/Llama-3.2-1B-Instruct',
                       adapter_id: str | None = 'AvaLovelace/BrickGPT', extra_adapter: str | Path | None = None,
                       tokenizer_id: str = 'AvaLovelace/BrickGPT') -> str:
    """Released BrickGPT = base + official LoRA, merged (as pilot/common.py load_model). If extra_adapter is
    given, that LoRA (trained on top of the merged model) is merged too. To merge only an extra adapter into an
    already-merged released model: base_id=<merged dir>, adapter_id=None, tokenizer_id=<merged dir>. Saved as a full model directory,
    because brickgpt.models.LLM loads with AutoModelForCausalLM.from_pretrained(path). Weights saved in fp32."""
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    out_dir = Path(out_dir)
    if (out_dir / 'config.json').exists():
        print(f'[build_merged_model] reuse {out_dir}')
        return str(out_dir)
    model = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.float32)
    if adapter_id is not None:
        model = PeftModel.from_pretrained(model, adapter_id).merge_and_unload()
    if extra_adapter is not None:
        model = PeftModel.from_pretrained(model, str(extra_adapter)).merge_and_unload()
    tok = AutoTokenizer.from_pretrained(tokenizer_id)
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    del model
    print(f'[build_merged_model] saved {out_dir}')
    return str(out_dir)


# ======================================================================================
# 4. Test subset + generation
# ======================================================================================
def load_test_subset(n: int, seed: int, dataset: str = 'AvaLovelace/StableText2Brick', split: str = 'test',
                     caption_index: int = 0) -> list[dict]:
    """Deduplicate the test split by object_id (as pilot/gen.py), then draw n objects without replacement
    with numpy RandomState(seed). Caption = captions[caption_index]."""
    from datasets import load_dataset
    df = load_dataset(dataset, split=split).to_pandas()
    df = df.drop_duplicates('object_id').reset_index(drop=True)
    rng = np.random.RandomState(seed)
    idx = sorted(rng.choice(len(df), size=min(n, len(df)), replace=False).tolist())
    return [{'item': k, 'row': int(i), 'object_id': str(df.loc[i, 'object_id']),
             'caption': str(df.loc[i, 'captions'][caption_index]), 'gt_bricks': str(df.loc[i, 'bricks'])}
            for k, i in enumerate(idx)]


def read_jsonl(path: str | Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        return []
    out = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f'WARNING: skipped bad line in {path}')
    return out


def append_jsonl(path: str | Path, rec: dict) -> None:
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())


def count_1x3(bricks) -> int:
    return sum(1 for b in bricks if (b.h, b.w) in [(1, 3), (3, 1)])


def run_generation(cfg_kwargs: dict, items: list[dict], out_path: str | Path, cond: str, seed_base: int,
                   instruction_fn=None, extra_fields: dict | None = None, bg=None):
    """Generate one structure per item with brickgpt.models.BrickGPT and append one JSONL line per item.
    Resumable: items already present in out_path for this `cond` are skipped.
    Seed for item k = seed_base + k (transformers.set_seed), identical across conditions.
    Returns the BrickGPT object so it can be reused (pass it back via bg= to avoid reloading the model)."""
    if TOL_FIX:
        patch_stability_tolerance()
    import torch
    import transformers
    from brickgpt.models import BrickGPT, BrickGPTConfig
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = {r['item'] for r in read_jsonl(out_path) if r.get('cond') == cond}
    todo = [it for it in items if it['item'] not in done]
    print(f'[{cond}] {len(done)} done, {len(todo)} to go -> {out_path}')
    from brickgpt.models import create_instruction as _orig_instruction
    cfg = BrickGPTConfig(**cfg_kwargs)
    if bg is None:
        bg = BrickGPT(cfg)
        bg._model_path = cfg.model_name_or_path
    else:  # reuse loaded model, update the generation settings
        assert getattr(bg, '_model_path', None) == cfg.model_name_or_path, 'bg was loaded from a different model'
        for k in ['world_dim', 'max_bricks', 'max_brick_rejections', 'use_logit_masking', 'max_regenerations',
                  'use_gurobi', 'temperature', 'temperature_increase', 'max_temperature', 'top_k', 'top_p']:
            setattr(bg, k, getattr(cfg, k))
    bg.instruction_fn = instruction_fn or _orig_instruction  # None -> original BrickGPT prompt
    for it in todo:
        seed = seed_base + it['item']
        transformers.set_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        err = None
        try:
            res = bg(it['caption'])
            bricks_txt = res['bricks'].to_txt()
            n_bricks = len(res['bricks'])
            n13 = count_1x3(res['bricks'].bricks)
            rej = dict(res['rejection_reasons'])
            n_regen = res['n_regenerations']
        except Exception as e:  # e.g. Gurobi licence error during rollback; keep going, record it
            bricks_txt, n_bricks, n13, rej, n_regen = '', 0, 0, {}, None
            err = f'{type(e).__name__}: {e}'[:500]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        rec = dict(cond=cond, item=it['item'], object_id=it['object_id'], caption=it['caption'], seed=seed,
                   n_bricks=n_bricks, n_1x3=n13, rejection_reasons=rej,
                   n_rejections=int(sum(rej.values())), n_regenerations=n_regen,
                   secs=round(time.time() - t0, 2), error=err, bricks=bricks_txt,
                   cfg={k: v for k, v in cfg_kwargs.items()}, **(extra_fields or {}))
        append_jsonl(out_path, rec)
        print(f'[{cond}] item {it["item"]}: {n_bricks} bricks, n_1x3={n13}, regen={n_regen}, '
              f'rej={rec["n_rejections"]}, {rec["secs"]}s' + (f', ERROR {err}' if err else ''), flush=True)
    return bg


# ======================================================================================
# 5. Scoring + metrics
# ======================================================================================
STAB_TOL = 1e-9
TOL_FIX = True  # apply patch_stability_tolerance() in scoring AND generation-time rollback
_tol_patched = False


def patch_stability_tolerance(tol: float = STAB_TOL) -> None:
    """Fix for a bug in BrickGPT's stability_analysis.py (see newbrick/investigate_score1/REPORT.md):
    a brick gets score 1.0 if any force/torque residual `.X > 0`; Gurobi often returns ~1e-14 float noise,
    which marks EVERY brick of a stable structure as 1.0 (~15% of dataset structures). Re-exec the original
    source with `.X > STAB_TOL` and rebind it where it was imported. Same as newbrick/tolerant_stability.py."""
    global _tol_patched
    if _tol_patched:
        return
    import importlib
    import sys
    mod = importlib.import_module('brickgpt.stability_analysis.stability_analysis')
    src = open(mod.__file__, encoding='utf-8').read()
    assert src.count('.X > 0') == 5, 'stability_analysis.py changed; check the tolerance patch'
    mod.__dict__['STAB_TOL'] = tol
    exec(compile(src.replace('.X > 0', '.X > STAB_TOL'), mod.__file__ + ' (tolerant)', 'exec'), mod.__dict__)
    for u in ['brickgpt.stability_analysis', 'brickgpt.data.brick_structure']:
        m = importlib.import_module(u)
        if hasattr(m, 'stability_score'):
            m.stability_score = mod.stability_score
    _tol_patched = True


def score_bricks_txt(bricks_txt: str, use_gurobi: bool, world_dim: int = 20) -> dict:
    """Post-hoc check of one generated structure (can be re-run locally on the saved JSONL).

    valid  := non-empty, parses, every brick in the CURRENT library, in bounds, no collisions.
    stable := valid and BrickStructure.is_stable() logic (no floating bricks, no collisions,
              max Gurobi stability score < 1). With use_gurobi=False, the connectivity check
              (BrickStructure.is_connected logic) is used instead and stored under *_conn keys.
    Per-brick score s_b = max of the analysis heat-map over the brick's voxels (>= 1 = brick fails);
    per-brick stability = 1 - min(s_b, 1); structure mean/min stability = mean/min over bricks.
    NOTE: this mean/min definition is ours; check it against the BrickGPT paper before comparing numbers.
    """
    from brickgpt.data import Brick, BrickStructure
    r = dict(valid=False, invalid_reason=None, stable=None, max_score=None,
             mean_stability=None, min_stability=None, checker='gurobi' if use_gurobi else 'connectivity',
             checker_error=None, check_secs=None)
    lines = [l for l in bricks_txt.split('\n') if l.strip()]
    if not lines:
        r['invalid_reason'] = 'empty'
        return r
    try:
        bricks = [Brick.from_txt(l) for l in lines]
    except ValueError:
        r['invalid_reason'] = 'ill_formatted'
        return r
    try:
        for b in bricks:
            _ = b.brick_id
    except ValueError:
        r['invalid_reason'] = 'not_in_library'
        return r
    bs = BrickStructure([], world_dim=world_dim)
    for b in bricks:
        if not bs.brick_in_bounds(b):
            r['invalid_reason'] = 'out_of_bounds'
            return r
        bs.add_brick(b)
    if bs.has_collisions():
        r['invalid_reason'] = 'collision'
        return r
    r['valid'] = True
    if TOL_FIX:
        patch_stability_tolerance()
    t0 = time.time()
    try:
        scores = bs.stability_scores() if use_gurobi else bs.connectivity_scores()
    except Exception as e:  # licence / solver errors are recorded, NOT counted as unstable
        r['checker_error'] = f'{type(e).__name__}: {e}'[:500]
        return r
    r['check_secs'] = round(time.time() - t0, 3)
    per_brick = np.array([float(np.max(scores[b.slice])) for b in bs.bricks])
    stab = 1.0 - np.minimum(per_brick, 1.0)
    r['max_score'] = float(per_brick.max())
    r['mean_stability'] = float(stab.mean())
    r['min_stability'] = float(stab.min())
    r['stable'] = bool((not bs.has_floating_bricks()) and per_brick.max() < 1)
    r['has_floating'] = bool(bs.has_floating_bricks())
    return r


def score_file(gen_path: str | Path, scored_path: str | Path, use_gurobi: bool, conds: list[str] | None = None):
    """Score every generated record not yet in scored_path (resumable). Scored line = generated line + scores."""
    gen = read_jsonl(gen_path)
    done = {(r['cond'], r['item']) for r in read_jsonl(scored_path)}
    for g in gen:
        if conds and g['cond'] not in conds:
            continue
        if (g['cond'], g['item']) in done:
            continue
        s = score_bricks_txt(g['bricks'], use_gurobi)
        append_jsonl(scored_path, {**g, **{'score_' + k: v for k, v in s.items()}})
    return read_jsonl(scored_path)


def _mean(xs):
    xs = [x for x in xs if x is not None]
    return float(np.mean(xs)) if xs else None


def summarize(scored: list[dict]) -> dict:
    """Per-condition metrics. Rates are over all items of the condition; generation errors and checker
    errors are counted separately (and excluded from the stability denominator)."""
    out = {}
    for cond in sorted({r['cond'] for r in scored}):
        rs = [r for r in scored if r['cond'] == cond]
        checked = [r for r in rs if r['score_valid'] and r['score_checker_error'] is None]
        valid_or_invalid = [r for r in rs if r['error'] is None]
        out[cond] = dict(
            n=len(rs),
            n_generation_errors=sum(r['error'] is not None for r in rs),
            n_checker_errors=sum(r['score_checker_error'] is not None for r in rs),
            checker=sorted({r['score_checker'] for r in rs}),
            validity_rate=_mean([float(r['score_valid']) for r in valid_or_invalid]),
            # stable rate over all non-error items (invalid -> not stable), as a fraction of generated structures
            stability_rate=(sum(bool(r['score_stable']) for r in checked) /
                            max(1, len([r for r in valid_or_invalid if r['score_checker_error'] is None]))),
            mean_stability=_mean([r['score_mean_stability'] for r in checked]),
            min_stability=_mean([r['score_min_stability'] for r in checked]),
            mean_regenerations=_mean([r['n_regenerations'] for r in rs]),
            mean_rejections=_mean([r['n_rejections'] for r in rs]),
            mean_n_bricks=_mean([r['n_bricks'] for r in rs]),
            mean_n_1x3=_mean([r['n_1x3'] for r in rs]),
            frac_with_1x3=_mean([float(r['n_1x3'] > 0) for r in rs]),
            mean_secs=_mean([r['secs'] for r in rs]),
            invalid_reasons=dict(Counter(r['score_invalid_reason'] for r in rs if r['score_invalid_reason'])),
        )
    return out


# ======================================================================================
# 6. LoRA training
# ======================================================================================
# Defaults = pilot/BrickGPT/scripts/finetune.zsh (original BrickGPT fine-tuning). Change in the notebook.
DEFAULT_HP = dict(
    lora_r=32, lora_alpha=16, lora_dropout=0.05, lora_target_modules=['q_proj', 'v_proj'],
    learning_rate=2e-3, lr_scheduler_type='cosine', warmup_steps=100, num_train_epochs=3,
    per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=4,
    max_length=8192, eval_steps=250, save_steps=500, load_best_model_at_end=True,
    precision='auto',            # 'auto' -> bf16 if the GPU supports it (A100) else fp16 (T4); or 'bf16'/'fp16'/'fp32'
    loss_on='all',               # 'all' tokens (what `trl sft` does on a "messages" dataset by default, we believe)
                                 # or 'assistant' (prompt tokens masked, as pilot/lora_tiny.py)
    gradient_checkpointing=False,
    dataloader_num_workers=2,    # finetune.zsh used 4; Colab has 2 CPU cores
    seed=42,
)


def load_messages_jsonl(path: str | Path, instruction_fn=None) -> list[list[dict]]:
    """Accepts {"messages": [...]} lines (stage2_data / prepare_finetuning_dataset.py format). Lines with
    {"caption"/"captions", "bricks"} are converted with instruction_fn (one example per caption)."""
    out = []
    for r in read_jsonl(path):
        if 'messages' in r:
            out.append(r['messages'])
        elif 'bricks' in r:
            fn = instruction_fn or (lambda c: create_instruction(c, NEW_DIMS))
            caps = r.get('captions') or [r['caption']]
            for c in caps:
                out.append([{'role': 'system', 'content': SYSTEM_PROMPT},
                            {'role': 'user', 'content': fn(c)},
                            {'role': 'assistant', 'content': r['bricks']}])
        else:
            raise ValueError(f'Unknown record format in {path}: keys {list(r)}')
    return out


def tokenize_messages(tok, messages: list[dict], max_length: int, loss_on: str) -> dict:
    """Chat-template the full conversation (like TRL does). NOTE: the Llama-3 template strips the assistant
    content, so the final '\\n' of the brick list is dropped before <|eot_id|> -- same as the original
    fine-tuning, which also went through apply_chat_template."""
    ids = tok.apply_chat_template(messages, tokenize=True)
    if hasattr(ids, 'keys'):  # some transformers versions return a BatchEncoding
        ids = ids['input_ids']
    ids = list(ids)[:max_length]
    labels = list(ids)
    if loss_on == 'assistant':
        prompt = tok.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
        if hasattr(prompt, 'keys'):
            prompt = prompt['input_ids']
        n = len(prompt)
        assert ids[:n] == list(prompt)[:len(ids[:n])], 'prompt is not a prefix of the full conversation'
        labels[:n] = [-100] * min(n, len(labels))
    elif loss_on != 'all':
        raise ValueError(loss_on)
    return {'input_ids': ids, 'labels': labels}


class _Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, feats):
        import torch
        L = max(len(f['input_ids']) for f in feats)
        ids = torch.full((len(feats), L), self.pad_id, dtype=torch.long)
        lab = torch.full((len(feats), L), -100, dtype=torch.long)
        att = torch.zeros((len(feats), L), dtype=torch.long)
        for i, f in enumerate(feats):
            n = len(f['input_ids'])
            ids[i, :n] = torch.tensor(f['input_ids'])
            lab[i, :n] = torch.tensor(f['labels'])
            att[i, :n] = 1
        return {'input_ids': ids, 'labels': lab, 'attention_mask': att}


class _ListDataset:
    def __init__(self, xs):
        self.xs = xs

    def __len__(self):
        return len(self.xs)

    def __getitem__(self, i):
        return self.xs[i]


def train_lora(merged_dir: str, train_path: str | Path, out_dir: str | Path, hp: dict,
               val_path: str | Path | None = None, max_steps: int = -1, run_name: str | None = None) -> dict:
    """Train a fresh LoRA on top of the merged released BrickGPT (as pilot/lora_tiny.py, but with HF Trainer
    and the finetune.zsh hyper-parameters). Saves the best (by eval loss, if val_path) adapter to out_dir and
    a JSON log (hp, dataset sizes, log_history, versions, wall time) to out_dir/train_log.json."""
    import torch
    import transformers
    import peft
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

    hp = {**DEFAULT_HP, **hp}
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    transformers.set_seed(hp['seed'])
    random.seed(hp['seed'])

    tok = AutoTokenizer.from_pretrained(merged_dir)
    if tok.pad_token is None or tok.pad_token_id == tok.eos_token_id:
        # finetuning_config_files: pad must differ from eos so the model learns to emit <|eot_id|>
        tok.pad_token = '<|finetune_right_pad_id|>'
    train_msgs = load_messages_jsonl(train_path)
    train_ds = _ListDataset([tokenize_messages(tok, m, hp['max_length'], hp['loss_on']) for m in train_msgs])
    val_ds = None
    if val_path and Path(val_path).exists():
        val_ds = _ListDataset([tokenize_messages(tok, m, hp['max_length'], hp['loss_on'])
                               for m in load_messages_jsonl(val_path)])

    prec = hp['precision']
    if prec == 'auto':
        prec = 'bf16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (
            'fp16' if torch.cuda.is_available() else 'fp32')
    # Master weights in fp32 (LoRA params must be fp32 for the fp16 GradScaler); autocast does the rest.
    model = AutoModelForCausalLM.from_pretrained(merged_dir, torch_dtype=torch.float32)
    if hp['gradient_checkpointing']:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=hp['lora_r'], lora_alpha=hp['lora_alpha'],
                                             lora_dropout=hp['lora_dropout'],
                                             target_modules=list(hp['lora_target_modules']),
                                             task_type='CAUSAL_LM'))
    model.print_trainable_parameters()

    use_eval = val_ds is not None
    args = TrainingArguments(
        output_dir=str(out_dir / 'checkpoints'),
        run_name=run_name,
        per_device_train_batch_size=hp['per_device_train_batch_size'],
        per_device_eval_batch_size=hp['per_device_eval_batch_size'],
        gradient_accumulation_steps=hp['gradient_accumulation_steps'],
        learning_rate=hp['learning_rate'],
        lr_scheduler_type=hp['lr_scheduler_type'],
        warmup_steps=hp['warmup_steps'],
        num_train_epochs=hp['num_train_epochs'],
        max_steps=max_steps,
        eval_strategy='steps' if use_eval else 'no',
        eval_steps=hp['eval_steps'],
        save_strategy='steps',
        save_steps=hp['save_steps'],
        save_total_limit=2,
        load_best_model_at_end=bool(hp['load_best_model_at_end'] and use_eval),
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=10,
        bf16=(prec == 'bf16'),
        fp16=(prec == 'fp16'),
        dataloader_num_workers=hp['dataloader_num_workers'],
        report_to='none',
        seed=hp['seed'],
        remove_unused_columns=False,
        use_cpu=not torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=_Collator(tok.pad_token_id))
    t0 = time.time()
    trainer.train()
    secs = time.time() - t0
    final_eval = trainer.evaluate() if use_eval else None
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    log = dict(run_name=run_name, train_path=str(train_path), val_path=str(val_path) if val_path else None,
               n_train_examples=len(train_ds), n_val_examples=len(val_ds) if val_ds else 0,
               mean_train_tokens=float(np.mean([len(x['input_ids']) for x in train_ds.xs])),
               n_truncated=sum(len(x['input_ids']) >= hp['max_length'] for x in train_ds.xs),
               hp=hp, precision_used=prec, max_steps=max_steps, train_secs=round(secs, 1),
               global_steps=trainer.state.global_step, best_checkpoint=trainer.state.best_model_checkpoint,
               final_eval=final_eval, log_history=trainer.state.log_history,
               versions=dict(torch=torch.__version__, transformers=transformers.__version__,
                             peft=peft.__version__),
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    with open(out_dir / 'train_log.json', 'w', encoding='utf-8') as f:
        json.dump(log, f, indent=1, default=str)
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return log

In [ ]:
sys.path[:0] = ['/content', '/content/BrickGPT/src']
import importlib, json
import pandas as pd
import torch
import helpers as H
importlib.reload(H)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert not H.library_has_1x3(), 'library must start without the 1x3 brick'

## Gurobi (WLS academic licence)

BrickGPT's physics stability check (and physics-informed rollback with `use_gurobi=True`) needs Gurobi with a full
licence; the pip `gurobipy` restricted licence fails with "Model too large" on real structures (we saw this
locally on 10/10 test structures).

* Put your `gurobi.lic` (WLS type) at `GUROBI_LIC_DRIVE_PATH` on Drive, or upload it when prompted below.
  **Never paste the licence contents into the notebook.** It is copied to `/content/gurobi.lic` (not saved to Drive
  by this notebook) and `GRB_LICENSE_FILE` is pointed at it.
* **Uncertain:** whether the Gurobi academic WLS licence terms allow use on Google Colab (cloud VM, not a
  university machine) is something we have *not* verified — check the licence terms / ask Gurobi before relying on it.
  Also unknown: whether the WLS licence limits concurrent sessions.
* **Fallback:** set `USE_GUROBI = False`. Then (a) rollback ON uses BrickGPT's simpler connectivity check during
  generation (a different method — the condition is renamed `rollback_on_conn`), and (b) post-hoc scoring uses
  connectivity, which is **not** a substitute for Min/Mean Stability. All generated bricks are saved, so Gurobi
  stability can be re-scored later on a machine with the licence (see the last section). Rollback OFF never calls a
  stability checker during generation, so its outputs are identical either way.

In [ ]:
if USE_GUROBI:
    import shutil
    if os.path.exists(GUROBI_LIC_DRIVE_PATH):
        shutil.copy(GUROBI_LIC_DRIVE_PATH, '/content/gurobi.lic')
        print('copied licence from Drive')
    elif not os.path.exists('/content/gurobi.lic'):
        from google.colab import files
        print('Upload gurobi.lic (WLS academic licence file)')
        up = files.upload()
        name = next(iter(up))
        os.replace(name, '/content/gurobi.lic')
    os.environ['GRB_LICENSE_FILE'] = '/content/gurobi.lic'
    import gurobipy as gp
    try:
        m = gp.Model('licence_test'); x = m.addVars(3000); m.update(); m.optimize()  # >2000 vars: fails on the restricted licence
        print('Gurobi OK:', gp.gurobi.version())
    except gp.GurobiError as e:
        print('Gurobi FAILED:', e, '\n-> fix the licence or set USE_GUROBI = False and re-run from the PARAMETERS cell')
        raise
else:
    print('USE_GUROBI=False: connectivity check only; re-score stability locally later.')

In [ ]:
# Released BrickGPT, merged into one full model directory (brickgpt.models.LLM loads a full model path)
RELEASED_DIR = H.build_merged_model(f'{MODELS_DIR}/brickgpt-merged', base_id=BASE_MODEL, adapter_id=BRICKGPT_ADAPTER)

# Stage 0 — baseline (original brick library, original prompt)

**How rollback is switched off** (from `pilot/BrickGPT/src/brickgpt/models/brickgpt.py`, `BrickGPT.__call__`):

```python
for regeneration_num in range(self.max_regenerations + 1):
    bricks, ... = self._generate_structure(caption, starting_bricks=starting_bricks)
    if self.max_regenerations == 0 or self._is_stable(bricks):
        break
    ...
    starting_bricks = self._remove_all_bricks_after_first_unstable_brick(bricks)
```

* `max_regenerations=0` → the loop runs once and breaks *before* `_is_stable` is evaluated (short-circuit `or`),
  so no stability check and no rollback happens at all; `n_regenerations` is 0. The config help text says the same:
  "Set to 0 if you want to disable physics-informed rollback." `use_gurobi` is then irrelevant during generation.
* Rollback ON = default `max_regenerations=100`: after each full generation the structure is checked
  (`is_stable()` with Gurobi, or `is_connected()` if `use_gurobi=False`); if unstable, all bricks from the first
  unstable brick on are removed and generation continues from the prefix, up to 100 times.
* Independent of rollback, **per-brick rejection sampling** (`max_brick_rejections=500`) rejects bricks that are
  ill-formatted, not in the library, out of bounds or colliding, and **logit masking** (`use_logit_masking=True`)
  enforces the `hxw (x,y,z)` syntax. Both stay ON in both conditions (the "no physics rollback" ablation). Setting
  `ALSO_DISABLE_REJECTION_IN_OFF=True` additionally sets `max_brick_rejections=0` (then invalid bricks are kept).

**Metrics** (computed post hoc from the saved bricks by `helpers.score_bricks_txt`, so they can be recomputed):
validity rate (non-empty, all bricks in library, in bounds, no collisions), stability rate (valid and no floating
bricks and every brick's Gurobi score < 1, i.e. `BrickStructure.is_stable()`), mean / min stability (per-brick
stability = 1 − min(score, 1), mean/min over bricks, averaged over valid structures — *our* definition; check against
the paper before quoting), mean number of regenerations, plus rejections, brick counts and time per item.

In [ ]:
S0 = f'{OUT}/stage0'; os.makedirs(S0, exist_ok=True)
subset_path = f'{S0}/subset.json'
if os.path.exists(subset_path):
    subset = json.load(open(subset_path))
    assert subset['seed'] == SUBSET_SEED and subset['n'] == N_CAPTIONS, 'subset.json differs from PARAMETERS'
    ITEMS = subset['items']
else:
    ITEMS = H.load_test_subset(N_CAPTIONS, SUBSET_SEED, dataset=DATASET, caption_index=CAPTION_INDEX)
    json.dump(dict(dataset=DATASET, split='test', dedup='object_id', seed=SUBSET_SEED, n=N_CAPTIONS,
                   caption_index=CAPTION_INDEX, items=ITEMS), open(subset_path, 'w'), indent=1)
print(len(ITEMS), 'items; first:', ITEMS[0]['caption'])

def cond_cfg(cond, model_dir):
    c = dict(model_name_or_path=model_dir, max_bricks=MAX_BRICKS, max_brick_rejections=MAX_BRICK_REJECTIONS,
             use_gurobi=USE_GUROBI, device='cuda' if torch.cuda.is_available() else 'cpu')
    if cond == 'rollback_on':
        c['max_regenerations'] = MAX_REGENERATIONS_ON
    elif cond == 'rollback_off':
        c['max_regenerations'] = 0
        if ALSO_DISABLE_REJECTION_IN_OFF:
            c['max_brick_rejections'] = 0
    else:
        raise ValueError(cond)
    return c

def cond_name(cond):  # make the stability checker used for rollback explicit in the saved data
    return cond + ('_conn' if (cond == 'rollback_on' and not USE_GUROBI) else '')

In [ ]:
# Timing pilot: TIMING_N items per condition, written to a separate file. Use it to extrapolate before the full run.
bg = None
for cond in CONDITIONS:
    bg = H.run_generation(cond_cfg(cond, RELEASED_DIR), ITEMS[:TIMING_N], f'{S0}/timing.jsonl', cond_name(cond),
                          GEN_SEED_BASE, bg=bg)
for cond in CONDITIONS:
    rs = [r for r in H.read_jsonl(f'{S0}/timing.jsonl') if r['cond'] == cond_name(cond)]
    secs = [r['secs'] for r in rs]
    print(f'{cond_name(cond)}: secs/item {secs} -> x{len(ITEMS)} items ~ {sum(secs)/len(secs)*len(ITEMS)/3600:.1f} h '
          f'(very rough: time depends strongly on brick count and regenerations)')

In [ ]:
# Stage 0 full run (resumable)
for cond in CONDITIONS:
    bg = H.run_generation(cond_cfg(cond, RELEASED_DIR), ITEMS, f'{S0}/gen.jsonl', cond_name(cond),
                          GEN_SEED_BASE, bg=bg)
del bg; torch.cuda.empty_cache()

In [ ]:
# Scoring + metrics (resumable; Gurobi if USE_GUROBI else connectivity)
scored = H.score_file(f'{S0}/gen.jsonl', f'{S0}/scored_{"gurobi" if USE_GUROBI else "conn"}.jsonl', USE_GUROBI)
m0 = H.summarize(scored)
json.dump(dict(metrics=m0, params=dict(N_CAPTIONS=N_CAPTIONS, SUBSET_SEED=SUBSET_SEED, GEN_SEED_BASE=GEN_SEED_BASE,
               MAX_BRICKS=MAX_BRICKS, MAX_REGENERATIONS_ON=MAX_REGENERATIONS_ON, USE_GUROBI=USE_GUROBI)),
          open(f'{S0}/metrics_{"gurobi" if USE_GUROBI else "conn"}.json', 'w'), indent=1)
pd.DataFrame(m0).T

# Stage 3 — LoRA on the Stage-2 data

**Data:** upload the folder written by `newbrick/build_stage2_data.py` (`newbrick/stage2_data/`) to `STAGE2_DIR`
on Drive. Format (same as BrickGPT's `prepare_finetuning_dataset.py`): one JSON line per (shape, caption),
`{"messages": [system "You are a helpful assistant.", user create_instruction(caption) with "1x3, 3x1" added to the
allowed list, assistant "hxw (x,y,z)\n..."]}`. Files `{B_best,C1,C2}_N{100,500,2000}.jsonl` and
`heldout_B_best.jsonl` (used here as the validation set). `helpers.load_messages_jsonl` also accepts
`{"captions"/"caption", "bricks"}` lines.

**Model:** a *fresh* LoRA on top of the merged released BrickGPT (like `pilot/lora_tiny.py`); the adapter is saved
and merged again for evaluation.

**Hyper-parameters** (defaults = `pilot/BrickGPT/scripts/finetune.zsh`, the original BrickGPT fine-tuning, run with
`trl sft`): LoRA r=32, alpha=16, dropout=0.05, target q_proj,v_proj; lr 2e-3, cosine, 100 warm-up steps, 3 epochs;
batch 2 × grad-accum 4 (effective 8); max_length 8192; eval every 250 steps, save every 500, load best at end; bf16.
Identical for every group/N, overridable via `HP_OVERRIDES`, logged to `train_log.json` in each run dir.
Differences from the original (unavoidable or deliberate, all logged):
* We use HF `Trainer` with our own tokenisation (`apply_chat_template` on the whole conversation) instead of `trl sft`,
  to avoid TRL API drift. `loss_on='all'` trains on all tokens, which we *believe* is what `trl sft` 0.17 does for a
  `messages` dataset without a completion-only collator (not verified); `loss_on='assistant'` masks the prompt.
* T4 has no bf16 → `precision='auto'` uses fp16 autocast on T4 and bf16 on A100 (fp32 master weights in both).
* `dataloader_num_workers` 2 instead of 4 (Colab CPU count).
* Starting point is the released BrickGPT (not raw Llama), and the dataset is much smaller: with N shapes × 5
  captions and effective batch 8, N=100 gives about 63 optimizer steps per epoch, so `warmup_steps=100` covers
  more than half of training and `eval_steps=250` may never trigger (then the final adapter is kept). Decide whether
  to keep the original values (comparability) or scale them (sensible schedule) — and use the same for all groups.

In [ ]:
S3 = f'{OUT}/stage3'; os.makedirs(S3, exist_ok=True)
HP = {**H.DEFAULT_HP, **HP_OVERRIDES}
json.dump(HP, open(f'{S3}/hyperparameters.json', 'w'), indent=1)
print(json.dumps(HP, indent=1))
RUNS = [(f'{g}_N{n}', f'{STAGE2_DIR}/{g}_N{n}.jsonl') for g in TRAIN_GROUPS for n in TRAIN_SIZES]
missing = [p for _, p in RUNS if not os.path.exists(p)]
print('runs:', [r for r, _ in RUNS]); print('MISSING files:', missing)
val_path = f'{STAGE2_DIR}/{VAL_FILE}' if VAL_FILE else None
if os.path.exists(f'{STAGE2_DIR}/manifest.json'):
    print(json.dumps(json.load(open(f'{STAGE2_DIR}/manifest.json'))['files'], indent=1)[:3000])

In [ ]:
for run, path in RUNS:
    out_dir = f'{S3}/{run}'
    if os.path.exists(f'{out_dir}/train_log.json'):
        print('skip (done):', run); continue
    if not os.path.exists(path):
        print('skip (no data):', path); continue
    print('=== training', run)
    log = H.train_lora(RELEASED_DIR, path, out_dir, HP, val_path=val_path, max_steps=MAX_STEPS, run_name=run)
    print(run, {k: log[k] for k in ['n_train_examples', 'global_steps', 'train_secs', 'final_eval', 'precision_used']})

# Stage 4 — evaluation with the trained adapters (1x3 brick in the library)

Same subset, seeds and conditions as Stage 0. The 1x3 brick (id 99, LDraw part 3622.DAT) is added to the in-memory
library (`helpers.set_1x3_in_library`, the patch from `pilot/common.py`), so generated 1x3/3x1 bricks are not rejected
as `not_in_library`. The prompt is taken from the adapter's own training file (everything before `### Input:\n`), so
evaluation uses exactly the training prompt. Control `released_1x3`: the released model with the same library and the
new prompt (B_best prompt), no new LoRA.

After a runtime restart, re-run the setup cells, the Gurobi cell and the first Stage-0 cell (it reloads `subset.json`
and defines `cond_cfg`) before this section; finished items are skipped.

In [ ]:
S4 = f'{OUT}/stage4'; os.makedirs(S4, exist_ok=True)
H.set_1x3_in_library(True, mass=NEW_BRICK_MASS)
assert H.library_has_1x3()
runs = EVAL_RUNS or sorted(d for d in os.listdir(S3) if os.path.exists(f'{S3}/{d}/train_log.json'))
evals = [(r, f'{S3}/{r}', f'{STAGE2_DIR}/{r}.jsonl') for r in runs]
if EVAL_CONTROL_RELEASED_WITH_1X3:
    ref = next((p for _, _, p in evals if os.path.exists(p)), f'{STAGE2_DIR}/B_best_N100.jsonl')
    evals = [('released_1x3', None, ref)] + evals
for name, adapter, data_path in evals:
    model_dir = RELEASED_DIR if adapter is None else H.build_merged_model(
        f'{MODELS_DIR}/eval-{name}', base_id=RELEASED_DIR, adapter_id=None, extra_adapter=adapter,
        tokenizer_id=RELEASED_DIR)
    fn = H.instruction_fn_from_prefix(H.prefix_from_dataset(data_path))
    d = f'{S4}/{name}'; os.makedirs(d, exist_ok=True)
    bg = None
    for cond in CONDITIONS:
        bg = H.run_generation(cond_cfg(cond, model_dir), ITEMS, f'{d}/gen.jsonl', cond_name(cond), GEN_SEED_BASE,
                              instruction_fn=fn, bg=bg, extra_fields=dict(run=name, library_1x3=True))
    del bg; torch.cuda.empty_cache()
    if adapter is not None:  # free local disk; the adapter itself stays on Drive
        import shutil; shutil.rmtree(model_dir, ignore_errors=True)
    tag = 'gurobi' if USE_GUROBI else 'conn'
    scored = H.score_file(f'{d}/gen.jsonl', f'{d}/scored_{tag}.jsonl', USE_GUROBI)
    json.dump(H.summarize(scored), open(f'{d}/metrics_{tag}.json', 'w'), indent=1)

In [ ]:
# Summary table: Stage 0 baseline + every Stage-4 run
tag = 'gurobi' if USE_GUROBI else 'conn'
rows = {}
for cond, m in json.load(open(f'{S0}/metrics_{tag}.json'))['metrics'].items():
    rows[('baseline', cond)] = m
for name in sorted(os.listdir(S4)):
    p = f'{S4}/{name}/metrics_{tag}.json'
    if os.path.exists(p):
        for cond, m in json.load(open(p)).items():
            rows[(name, cond)] = m
summary = pd.DataFrame(rows).T
summary.to_csv(f'{S4}/summary_{tag}.csv')
cols = ['n', 'validity_rate', 'stability_rate', 'mean_stability', 'min_stability', 'mean_regenerations',
        'mean_n_bricks', 'mean_n_1x3', 'frac_with_1x3', 'mean_secs', 'n_generation_errors', 'n_checker_errors']
summary[cols]

## Re-scoring stability locally (e.g. after a `USE_GUROBI=False` run)

All generated structures are in `gen.jsonl` (field `bricks`). On a machine with Gurobi and the licence, with our repo:

```python
import sys; sys.path[:0] = ['pilot/BrickGPT/src', 'newbrick/colab']
import helpers as H
H.set_1x3_in_library(True)            # only for Stage-4 files (harmless for Stage 0: no 1x3 bricks there)
scored = H.score_file('stage0/gen.jsonl', 'stage0/scored_gurobi.jsonl', use_gurobi=True)
print(H.summarize(scored))
```

Caveat: this re-scores the final structures; it cannot turn a connectivity-based rollback (`rollback_on_conn`)
into a Gurobi-based one — that requires regenerating with `USE_GUROBI=True`.